In [1]:
from nlp4bia.datasets.benchmark.symptemist import SymptemistLoader, SymptemistGazetteer
from sentence_transformers import SentenceTransformer
from nlp4bia.linking import BECELinker

# 1) Load data
df = SymptemistLoader().df
df_gaz = SymptemistGazetteer().df#.iloc[:100]

# biencoder_path = "/gpfs/projects/bsc14/abecerr1/hub/models--ICB-UMA--ClinLinker-KB-GP/snapshots/8f914c58a1cbcff43331eb15b101eaa5e5c6920a"
# biencoder_path = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/procedimiento/biencoder_medprocner_1_epoch_32_batch_5_parents_stag"
biencoder_path = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/sintoma/symptemist/biencoder_symptemist_1_epoch_64_batch_5_parents_stag"
# ce_path = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/procedimiento/crossencoder_medprocner_5_epoch_16_batch"
ce_path = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/sintoma/symptemist/crossencoder_symptemist_5_epoch_32_batch"

# # 2) Prepare a SentenceTransformer bi-encoder (already loaded)
# biencoder_model = SentenceTransformer(biencoder_path)
biencoder_model = SentenceTransformer(biencoder_path, device="cuda")
vector_db = biencoder_model.encode(
    df_gaz["term"].tolist(),
    batch_size=4096,
    show_progress_bar=True,
    convert_to_tensor=True
)


/gpfs/projects/bsc14/code/nlp4bia/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 41/41 [00:13<00:00,  2.96it/s]


In [2]:

# 3) Initialize the BECELinker
linker = BECELinker(
    df_gazetteer=df_gaz,
    biencoder_model_or_path=biencoder_path,
    crossencoder_model_or_path=ce_path,
    biencoder_batch_size=4096,
    reranker_batch_size=4096,
    vector_db=vector_db,
)

# 4) Link a list of mentions
ls_mentions = df["span"].tolist()[:10]
results = linker.link(
    mentions=ls_mentions,
    n_candidates=200,
    top_k=5,
    return_documents=True
)

# 5) Inspect output
for res in results:
    print(f"Mention: {res['mention']}")
    for idx, (term, code, score) in enumerate(zip(res["terms"], res["codes"], res["similarity"]), start=1):
        print(f"  {idx:02d}. {term} ({code}) → {score:.4f}")
    print()


Initializing DenseRetriever...
Using bi-encoder model: /gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/sintoma/symptemist/biencoder_symptemist_1_epoch_64_batch_5_parents_stag
Note: Vector DB will be computed on the fly. Increase `biencoder_batch_size` to accelerate this.
In case of MemoryError, try reducing `biencoder_batch_size` or using a smaller model.
DenseRetriever initialized successfully.
Initializing CrossEncoder Reranker...
Using CrossEncoder model: /gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/sintoma/symptemist/crossencoder_symptemist_5_epoch_32_batch


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.42it/s]

Mention: «manchas» en el campo visual
  01. observación de una mancha oscura en el campo visual (246658005) → 0.9834
  02. moscas volantes en el campo visual (162278001) → 0.9049
  03. aspecto ocular anormal (737269000) → 0.0422
  04. manchas retinianas (247138002) → 0.0289
  05. ve manchas frente a los ojos (418008001) → 0.0201

Mention: 5HIAA en orina de 24 horas estaba dentro de los parámetros normales
  01. excreción de 5 - HIAA en orina de 24 horas (313824003) → 0.9896
  02. medición de la excreción de 5 - HIAA en orina de 24 horas (313824003) → 0.9891
  03. rastreo en orina: normal (171250001) → 0.9372
  04. medición de AHV en orina (271254009) → 0.8804
  05. pesquisa en orina: normal (171250001) → 0.7141

Mention: A nivel analítico no presentaba alteración
  01. análisis bioquímico de la sangre normal (166315009) → 0.9882
  02. hallazgo bioquímico negativo (309305008) → 0.3962
  03. hallazgo relacionado con medición dentro del rango de referencia (442082004) → 0.0107
  04. sin h

In [ ]:
# Upload model to hub
# linker.retriever.model.push_to_hub("BSC-NLP4BIA/Symptemist-Biencoder",
#                                   token="",
#                                   private=False)

model.safetensors: 100%|██████████| 504M/504M [00:15<00:00, 32.0MB/s] 


'https://huggingface.co/BSC-NLP4BIA/Symptemist-Biencoder/commit/c5ffbff8bf673b6844271c00457708c8c906f020'

In [ ]:
# # Upload model to hub
# linker.reranker.model.push_to_hub("BSC-NLP4BIA/Symptemist-CE-Reranker",
#                                   token="",
#                                   private=False)

model.safetensors: 100%|██████████| 504M/504M [00:12<00:00, 41.1MB/s] 


'https://huggingface.co/BSC-NLP4BIA/Symptemist-CE-Reranker/commit/7c8c25200c163ea0a2a8716865d461ce430e3566'